# Lab 01.2 — customClaims e allowedScopes

## Overview

No notebook anterior criamos o User Pool e os grupos. Agora vamos:

1. Criar o **app client** que o portal usa para fazer login
2. Criar o **resource server** com scopes (`workshop/gateway/invoke`, etc.)
3. **Autenticar** Ana e Carlos para gerar JWTs reais
4. **Decodificar** os JWTs e ver as claims (`cognito:groups`, `email`, etc.)
5. Entender como o **Gateway authorizer** vai usar `customClaims` e `allowedScopes`

> 💡 **`customClaims`** is o que diz ao Gateway *qual claim do JWT identifica o
> principal do Cedar*. No nosso caso: `cognito:groups`.
>  
> **`allowedScopes`** is o conjunto de scopes que o Gateway aceita — qualquer
> JWT precisa ter um deles para chamar tools.

## Tutorial Details

| Information | Details |
|---|---|
| Tutorial type | Interactive |
| AgentCore components | Identity, Gateway (early configuration) |
| Complexity | Easy |
| SDK | boto3 + PyJWT |
| Estimated time | 8 minutes |

## Prerequisites

- ✅ [Lab 01.1 — Create Cognito Pool with Groups](./01-create-cognito-pool-with-groups.ipynb) concluído

## Setup

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")

from shared.utils.config import load_config, save_config, get_region
from utils import (
    create_app_client_user_password,
    create_resource_server_with_scopes,
    get_bearer_token,
    decode_jwt,
)

cfg = load_config()
region = get_region()
pool_id = cfg["COGNITO_USER_POOL_ID"]
print(f"Pool ID: {pool_id}")

## Step 1: Criar o app client (USER_PASSWORD_AUTH)

Este is o app client que o portal Flask do Lab 09 vai usar para fazer login dos
users. Since this is a traditional web portal, we use `USER_PASSWORD_AUTH` instead
de PKCE.

> 💡 **App client vs Resource server.** App clients are **application
> identities** (frontends, agents, scripts). Resource servers are **API
> de APIs** que o app client quer acessar — eles definem os scopes.

In [ ]:
client_id = create_app_client_user_password(
    pool_id=pool_id,
    client_name="workshop-portal-client",
    region=region,
)
print(f"App client ID: {client_id}")

## Step 2: Criar o resource server com scopes

O Gateway is uma API protegida — precisa de um resource server. Os scopes
definem operações distintas (`read`, `invoke`). O Gateway no Lab 02 vai
listar exatamente quais scopes ele aceita.

In [ ]:
rs = create_resource_server_with_scopes(
    pool_id=pool_id,
    identifier="workshop/gateway",
    name="WorkshopGatewayResource",
    region=region,
)
print(f"Identifier: {rs['identifier']}")
print(f"Full scopes: {rs['full_scopes']}")

> ⚠️ **Observação importante sobre scopes custom e o fluxo deste workshop**
>
> Acabamos de criar um resource server com os scopes `workshop/gateway/read` e
> `workshop/gateway/invoke`. Porism, **estes scopes custom NÃO entram no access
> token when o login is feito via `USER_PASSWORD_AUTH`** (o fluxo que usamos nos
> labs). O Cognito só inclui scopes custom em tokens obtidos pelos fluxos OAuth2
> (`client_credentials` ou `authorization_code`), atraviss do endpoint
> `/oauth2/token` do domínio Cognito.
>
> Com `USER_PASSWORD_AUTH`, o access token carrega apenas o scope
> `aws.cognito.signin.user.admin`. Por isso o **Gateway** (Lab 02) is configurado
> com `allowedScopes=["aws.cognito.signin.user.admin"]`.
>
> **Where does authorization live then?** The scope here is just an *authentication
> gate* ("is a valid user of this pool"). Fine-grained authorization — who can
> chamar qual ferramenta — is feita by the **Cedar** (Lab 03), usando o claim
> `cognito:groups`. Um scope não consegue expressar regras como "operador lê a
> rede mas não aprova ordens de serviço"; o Cedar consegue.
>
> **Por que não usar scope custom de verdade?**
> - `client_credentials`: would issue scopes, but has no user → no
>   `cognito:groups` → quebraria toda a governança por grupo do Cedar.
> - `authorization_code`: traria scopes **e** grupos juntos, mas exige login
>   interativo no browser (redirect), inviável num fluxo programático de notebook.
>
> Ou seja: validar `aws.cognito.signin.user.admin` + autorizar no Cedar is a
> correct choice for this scenario (real user, non-interactive, group-based authz).
> O resource server fica aqui como demonstração conceitual de M2M/OAuth2.


## Step 3: Autenticar Ana e Carlos — visualizar JWT

We will fazer login programático de Ana e Carlos e olhar os tokens. A senha is
a mesma que definimos no Lab 01.1 (`Workshop@2025!`).

In [ ]:
ana_tokens = get_bearer_token(
    pool_id=pool_id,
    client_id=client_id,
    username="ana.operadora@workshop.local",
    password="Workshop@2025!",
    region=region,
)
print(f"Ana access_token: {ana_tokens['access_token'][:60]}...")

carlos_tokens = get_bearer_token(
    pool_id=pool_id,
    client_id=client_id,
    username="carlos.gestor@workshop.local",
    password="Workshop@2025!",
    region=region,
)
print(f"Carlos access_token: {carlos_tokens['access_token'][:60]}...")

## Step 4: Decodificar e comparar os JWTs

The tokens are self-contained — all the claims that the Gateway/Cedar need
are dentro do próprio JWT.

⚠️ Estamos decodificando **sem validar a assinatura** — só para visualização
didática. O Gateway vai validar de verdade no Lab 02.

In [ ]:
ana_claims = decode_jwt(ana_tokens["access_token"])
carlos_claims = decode_jwt(carlos_tokens["access_token"])

print("=== Ana ===")
print(f"  username:        {ana_claims.get('username')}")
print(f"  cognito:groups:  {ana_claims.get('cognito:groups')}")
print(f"  scope:           {ana_claims.get('scope')}")
print(f"  token_use:       {ana_claims.get('token_use')}")
print(f"  exp:             {ana_claims.get('exp')}")

print("\n=== Carlos ===")
print(f"  username:        {carlos_claims.get('username')}")
print(f"  cognito:groups:  {carlos_claims.get('cognito:groups')}")
print(f"  scope:           {carlos_claims.get('scope')}")

## Step 5: Como o Gateway vai usar isso (preview do Lab 02)

No Lab 02, ao criar o Gateway, we will passar uma `authorizerConfiguration`
similar a esta:

```python
authorizer_config = {
    "customJWTAuthorizer": {
        "discoveryUrl": cfg["COGNITO_DISCOVERY_URL"],
        "allowedScopes": ["workshop/gateway/invoke"],
        "customClaims": {
            "principal": "cognito:groups"  # <— aponta para a claim que vira o principal Cedar
        }
    }
}
```

- **`discoveryUrl`** — o Gateway baixa o JWKS dessa URL para validar assinatura
- **`allowedScopes`** — qualquer JWT precisa ter um desses scopes
- **`customClaims.principal`** — o Cedar Policy Engine vai usar a claim
  `cognito:groups` para decidir PERMIT/DENY

## Step 6: Persist IDs to config.env

In [ ]:
save_config({
    "COGNITO_CLIENT_ID": client_id,
    "COGNITO_RESOURCE_SERVER_ID": rs["identifier"],
})

## ✅ Validation

Confirme que o token de Ana **has** a claim `cognito:groups: ['operators']` e
que o de Carlos has `['managers', 'governance']`. Se algum estiver vazio, ela
não foi adicionada ao grupo no Lab 01.1.

```python
assert "operators" in ana_claims.get("cognito:groups", []), "Ana deveria estar em operators"
assert "managers" in carlos_claims.get("cognito:groups", []), "Carlos deveria estar em managers"
print("✓ Claims OK")
```

In [ ]:
assert "operators" in ana_claims.get("cognito:groups", []), "Ana deveria estar em operators"
assert "managers" in carlos_claims.get("cognito:groups", []), "Carlos deveria estar em managers"
print("✓ Claims OK — Identity is pronto para o Gateway")

## 🎓 What you learned

- App client vs resource server (identidades de apps vs identidades de APIs)
- Como gerar JWTs programaticamente via `admin_initiate_auth`
- Estrutura de um access token Cognito (claims, scopes, exp)
- Como o Gateway vai usar `discoveryUrl`, `allowedScopes` e `customClaims`

## Next

➡️ [Lab 02 — AgentCore Gateway](../02-AgentCore-Gateway/)

We will create o Gateway com JWT authorizer (usando a config que acabamos de
preparar) e adicionar Lambdas como targets.